In [176]:
from google.colab import files

In [177]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

print("Libraries imported successfully!")

Libraries imported successfully!


In [178]:
import pandas as pd

# Load files already uploaded to Colab
housing = pd.read_csv(
    "/content/housing_clean (2).csv",
    low_memory=False
)

transport = pd.read_csv(
    "/content/crash_clean_prelim (1).csv",
    low_memory=False
)

print("HOUSING DATASET")
print("Shape:", housing.shape)
print("Columns:", housing.columns.tolist())
display(housing.head())

print("\nTRANSPORTATION DATASET")
print("Shape:", transport.shape)
print("Columns:", transport.columns.tolist())
display(transport.head())

print("\nBoth datasets loaded successfully!")

HOUSING DATASET
Shape: (571131, 103)
Columns: ['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board', 'work_on_floor', 'apt_condo_no_s', 'applicant_professional_title', 'applicant_license', 'applicant_first_name', 'applicants_middle_initial', 'applicant_last_name', 'owner_s_business_name', 'applicant_street_name', 'city', 'state', 'zip', 'filing_representative_first_name', 'filing_representative_middle_initial', 'filing_representative_last_name', 'filing_representative_business_name', 'filing_representative_street_name', 'filing_representative_city', 'filing_representative_state', 'filing_representative_zip', 'sprinkler_work_type', 'plumbing_work_type', 'standpipe', 'antenna', 'sign', 'curb_cut', 'fence', 'scaffold', 'initial_cost', 'total_construction_floor_area', 'review_building_code', 'little_e', 'unmapped_cco_street', 'request_legalization', 'includes_permanent_removal', 'in_compliance_with_nycecc', 'exempt_from_nycec

,job_filing_number,filing_status,house_no,street_name,borough,block,lot,bin,commmunity_board,work_on_floor,...,filing_review_type,job_base,job_suffix,proposed_dwelling_units_outlier,proposed_no_of_stories_outlier,initial_cost_outlier,year,month,year_month,days_permit_to_signoff
0,M00000111-I1,LOC Issued,31,WALKER STREET,Manhattan,193,25,1079007,101.0,4th,...,Standard Plan Examination,M00000111,I1,False,False,False,2016,8,2016-08,532.0
1,B00000136-I1,LOC Issued,379,Lefferts Avenue,Brooklyn,1321,48,3393507,309.0,1st,...,Professional Certification,B00000136,I1,False,False,False,2016,9,2016-09,318.0
2,B00000139-I1,Permit Entire,48,Cedar Street,Brooklyn,3232,1,3429266,304.0,"7, 8",...,Professional Certification,B00000139,I1,False,False,False,2016,9,2016-09,NaN
3,B00000119-I1,Permit Entire,642,DRIGGS AVENUE,Brooklyn,2352,25,3321245,301.0,CEL,...,Professional Certification,B00000119,I1,False,False,False,2016,10,2016-10,NaN
4,B00000150-I1,LOC Issued,904,RUTLAND ROAD,Brooklyn,4606,1,3099712,317.0,"cel, 1",...,Professional Certification,B00000150,I1,False,False,False,2016,10,2016-10,818.0



TRANSPORTATION DATASET
Shape: (13309, 21)
Columns: ['open_data_crash_id', 'crash_date', 'crash_time', 'crash_hour', 'crash_weekday', 'borough', 'borough_source', 'zip', 'zip_source', 'latitude', 'longitude', 'total_severe_injuries', 'total_non_severe_injuries', 'total_fatalities', 'similar_fatalities_severe', 'is_highway', 'is_repeat_location', 'year', 'month', 'year_month', 'time_is_midnight']


,open_data_crash_id,crash_date,crash_time,crash_hour,crash_weekday,borough,borough_source,zip,zip_source,latitude,...,total_severe_injuries,total_non_severe_injuries,total_fatalities,similar_fatalities_severe,is_highway,is_repeat_location,year,month,year_month,time_is_midnight
0,4492364,2022-01-01,01:28,1,Saturday,Bronx,original,NaN,missing,40.858707,...,1.0,1.0,0.0,0.0,False,False,2022,1,2022-01,False
1,4491159,2022-01-01,04:33,4,Saturday,Bronx,original,NaN,missing,NaN,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False
2,4491487,2022-01-01,05:55,5,Saturday,Bronx,original,NaN,missing,40.871550,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False
3,4491223,2022-01-01,10:42,10,Saturday,Bronx,original,NaN,missing,40.823673,...,1.0,0.0,0.0,0.0,True,False,2022,1,2022-01,False
4,4491418,2022-01-01,20:55,20,Saturday,Bronx,original,NaN,missing,40.863945,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False



Both datasets loaded successfully!


In [179]:
# Check Member 1's cleaned Housing data
print("HOUSING CLEANING CHECK")

important_columns = [
    "job_base",
    "filing_date",
    "h_filings_total",
    "h_new_building"
]

for col in important_columns:
    if col in housing.columns:
        print(f"{col}: Found")
    else:
        print(f"{col}: Not found")

if "job_base" in housing.columns:
    print(
        "Duplicate job bases:",
        housing["job_base"].duplicated().sum()
    )

HOUSING CLEANING CHECK
job_base: Found
filing_date: Found
h_filings_total: Not found
h_new_building: Not found
Duplicate job bases: 0


## 1. Integration Strategy and Justification

### Objective

The objective of this stage is to integrate NYC housing development records from DOB NOW with transportation safety records from SIRTA.

The integration will help us explore whether areas with higher building activity also show different patterns in transportation safety records.

### Proposed Integration Method

The preferred method is to connect both datasets using ZIP code and month, provided both datasets contain reliable geographic and date information.

Before merging, each dataset will be aggregated separately to avoid multiplying records that share the same ZIP code and month.

### Why This Method?

- ZIP codes allow geographic comparisons across NYC.
- Months allow us to examine changes over time.
- Aggregation reduces unnecessary duplication and makes the datasets easier to compare.
- A merge indicator helps identify which geographic and time combinations appear in both sources.

### Limitations

The results can show associations but cannot establish that construction activity causes transportation safety incidents.

Differences in population, road usage, traffic volume, and reporting procedures may influence the results.


In [180]:
# Work on copies to preserve the original data
housing_df = housing.copy()
transport_df = transport.copy()

# Standardize column names
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

housing_df = standardize_columns(housing_df)
transport_df = standardize_columns(transport_df)

print("HOUSING COLUMNS")
print(housing_df.columns.tolist())

print("\nSIRTA COLUMNS")
print(transport_df.columns.tolist())

HOUSING COLUMNS
['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board', 'work_on_floor', 'apt_condo_no_s', 'applicant_professional_title', 'applicant_license', 'applicant_first_name', 'applicants_middle_initial', 'applicant_last_name', 'owner_s_business_name', 'applicant_street_name', 'city', 'state', 'zip', 'filing_representative_first_name', 'filing_representative_middle_initial', 'filing_representative_last_name', 'filing_representative_business_name', 'filing_representative_street_name', 'filing_representative_city', 'filing_representative_state', 'filing_representative_zip', 'sprinkler_work_type', 'plumbing_work_type', 'standpipe', 'antenna', 'sign', 'curb_cut', 'fence', 'scaffold', 'initial_cost', 'total_construction_floor_area', 'review_building_code', 'little_e', 'unmapped_cco_street', 'request_legalization', 'includes_permanent_removal', 'in_compliance_with_nycecc', 'exempt_from_nycecc', 'building_type', 'existing

In [181]:
# Integration columns confirmed from the cleaned datasets

housing_zip = "postcode"
housing_date = "filing_date"
housing_borough = "borough"

transport_zip = "zip"
transport_date = "crash_date"
transport_borough = "borough"

print("HOUSING")
print("ZIP:", housing_zip)
print("Date:", housing_date)
print("Borough:", housing_borough)

print("\nSIRTA")
print("ZIP:", transport_zip)
print("Date:", transport_date)
print("Borough:", transport_borough)

HOUSING
ZIP: postcode
Date: filing_date
Borough: borough

SIRTA
ZIP: zip
Date: crash_date
Borough: borough


In [182]:
# Stop if the required columns could not be identified
required = {
    "Housing ZIP": housing_zip,
    "Housing date": housing_date,
    "SIRTA ZIP": transport_zip,
    "SIRTA date": transport_date
}

missing = [
    name for name, value in required.items()
    if value is None
]

if missing:
    raise ValueError(
        "These required columns need manual mapping: "
        + ", ".join(missing)
    )

# Standardize ZIP codes
def clean_zip(series):
    return (
        series.astype("string")
        .str.strip()
        .str.extract(r"^(\d{5})(?:\.0)?$", expand=False)
    )

# Standardize boroughs
def clean_borough(series):
    mapping = {
        "MN": "Manhattan",
        "MANHATTAN": "Manhattan",
        "BK": "Brooklyn",
        "BROOKLYN": "Brooklyn",
        "BX": "Bronx",
        "BRONX": "Bronx",
        "QN": "Queens",
        "QUEENS": "Queens",
        "SI": "Staten Island",
        "STATEN ISLAND": "Staten Island"
    }

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .map(mapping)
        .astype("string")
    )

# Create standardized integration fields
housing_df["zip_code"] = clean_zip(housing_df[housing_zip])
transport_df["zip_code"] = clean_zip(transport_df[transport_zip])

# Parse mixed date formats correctly
housing_df["date"] = pd.to_datetime(
    housing_df[housing_date],
    format="mixed",
    errors="coerce"
)

transport_df["date"] = pd.to_datetime(
    transport_df[transport_date],
    format="mixed",
    errors="coerce"
)

print("Housing missing dates:", housing_df["date"].isna().sum())
print("SIRTA missing dates:", transport_df["date"].isna().sum())

housing_df["month"] = housing_df["date"].dt.to_period("M")
transport_df["month"] = transport_df["date"].dt.to_period("M")

housing_df["borough"] = (
    clean_borough(housing_df[housing_borough])
    if housing_borough else pd.Series(
        pd.NA, index=housing_df.index, dtype="string"
    )
)

transport_df["borough"] = (
    clean_borough(transport_df[transport_borough])
    if transport_borough else pd.Series(
        pd.NA, index=transport_df.index, dtype="string"
    )
)

# Examine key completeness
for name, df in [
    ("Housing", housing_df),
    ("SIRTA", transport_df)
]:
    print(f"\n{name} integration key quality")
    print("Missing ZIP:", df["zip_code"].isna().sum())
    print("Missing date:", df["date"].isna().sum())
    print("Missing borough:", df["borough"].isna().sum())
    print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nStandardization completed.")

Housing missing dates: 0
SIRTA missing dates: 0

Housing integration key quality
Missing ZIP: 1469
Missing date: 0
Missing borough: 0
Date range: 2016-08-04 20:00:00 to 2026-10-31 00:00:00

SIRTA integration key quality
Missing ZIP: 5237
Missing date: 0
Missing borough: 55
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

Standardization completed.


In [183]:
# Investigate missing Housing filing dates

print("Original missing filing dates:")
print(housing["filing_date"].isna().sum())

print("\nSample original filing dates:")
print(housing["filing_date"].dropna().head(15).tolist())

# Try parsing multiple date formats
dates_mixed = pd.to_datetime(
    housing["filing_date"],
    format="mixed",
    errors="coerce"
)

print("\nMissing dates after mixed-format parsing:")
print(dates_mixed.isna().sum())

print("\nDates successfully recovered compared with current parsing:")
print(
    housing_df["date"].isna().sum()
    - dates_mixed.isna().sum()
)

print("\nExamples of originally unparsed dates:")
display(
    housing.loc[
        housing_df["date"].isna()
        & dates_mixed.notna(),
        ["filing_date"]
    ].head(10)
)

Original missing filing dates:
0

Sample original filing dates:
['2016-08-04 20:00:00', '2016-09-14 20:00:00', '2016-09-27 20:00:00', '2016-10-13 20:00:00', '2016-10-16 20:00:00', '2016-11-17 19:00:00', '2016-11-17 19:00:00', '2017-01-19 19:00:00', '2017-01-19 19:00:00', '2017-01-31 19:00:00', '2017-03-14 20:00:00', '2017-05-10 20:00:00', '2017-06-14 20:00:00', '2017-07-05 20:00:00', '2017-07-05 20:00:00']

Missing dates after mixed-format parsing:
0

Dates successfully recovered compared with current parsing:
0

Examples of originally unparsed dates:


,filing_date


In [184]:
# Check integration readiness

print("HOUSING")
print("Total records:", len(housing_df))
print("Missing ZIP:", housing_df["zip_code"].isna().sum())
print("Missing dates:", housing_df["date"].isna().sum())
print("Date range:", housing_df["date"].min(),
      "to", housing_df["date"].max())

print("\nSIRTA")
print("Total records:", len(transport_df))
print("Missing ZIP:", transport_df["zip_code"].isna().sum())
print("Missing dates:", transport_df["date"].isna().sum())
print("Date range:", transport_df["date"].min(),
      "to", transport_df["date"].max())

housing_ready = housing_df.dropna(
    subset=["zip_code", "date"]
)

transport_ready = transport_df.dropna(
    subset=["zip_code", "date"]
)

print("\nHousing usable records:", len(housing_ready))
print("SIRTA usable records:", len(transport_ready))

shared_zips = (
    set(housing_ready["zip_code"])
    & set(transport_ready["zip_code"])
)

print("Number of shared ZIP codes:", len(shared_zips))

HOUSING
Total records: 571131
Missing ZIP: 1469
Missing dates: 0
Date range: 2016-08-04 20:00:00 to 2026-10-31 00:00:00

SIRTA
Total records: 13309
Missing ZIP: 5237
Missing dates: 0
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

Housing usable records: 569662
SIRTA usable records: 8072
Number of shared ZIP codes: 186


In [185]:
# Check whether ZIP 10000 exists in NYC's official ZCTA dataset

zcta_url = (
    "https://data.cityofnewyork.us/api/v3/views/"
    "35j5-n34v/export.csv?accessType=DOWNLOAD"
)

zcta = pd.read_csv(zcta_url, dtype=str)

print("ZCTA columns:", zcta.columns.tolist())

zip_column = next(
    (col for col in zcta.columns
     if col.lower().strip() in ["zip code", "zcta5"]),
    None
)

if zip_column is None:
    raise ValueError("Could not identify the ZCTA ZIP column.")

valid_zctas = set(zcta[zip_column].dropna().str.strip())

print("Number of official ZCTAs:", len(valid_zctas))
print("ZIP 10000 found:", "10000" in valid_zctas)

print("\nHousing ZIPs without matching ZCTAs:")
print(
    sorted(set(housing_df["zip_code"].dropna()) - valid_zctas)[:30]
)

print("\nSIRTA ZIPs without matching ZCTAs:")
print(
    sorted(set(transport_df["zip_code"].dropna()) - valid_zctas)[:30]
)

ZCTA columns: ['Zip Code', 'Land Area', 'Water Area', 'Center Latitude', 'Center Longitude', 'Internal Latitude', 'Internal Longitude', 'Geometry']
Number of official ZCTAs: 221
ZIP 10000 found: False

Housing ZIPs without matching ZCTAs:
['10000', '10041', '10055', '10104', '10105', '10106', '10107', '10118', '10120', '10121', '10122', '10123', '10151', '10155', '10158', '10175', '10176', '10178', '10281', '11111', '11251']

SIRTA ZIPs without matching ZCTAs:
['10000', '10281', '11251']


In [186]:
# Flag unverified ZIP codes before integration

housing_df["zip_not_in_zcta"] = (
    housing_df["zip_code"].notna()
    & ~housing_df["zip_code"].isin(valid_zctas)
)

transport_df["zip_not_in_zcta"] = (
    transport_df["zip_code"].notna()
    & ~transport_df["zip_code"].isin(valid_zctas)
)

# Flag ZIP 10000 separately
housing_df["zip_10000_flag"] = (
    housing_df["zip_code"] == "10000"
).fillna(False)

transport_df["zip_10000_flag"] = (
    transport_df["zip_code"] == "10000"
).fillna(False)

print("Housing records with ZIP 10000:",
      housing_df["zip_10000_flag"].sum())

print("SIRTA records with ZIP 10000:",
      transport_df["zip_10000_flag"].sum())

print("\nZIP values absent from the ZCTA reference are flagged.")

Housing records with ZIP 10000: 163
SIRTA records with ZIP 10000: 44

ZIP values absent from the ZCTA reference are flagged.


## 2. Data Aggregation

### Data Aggregation

Before integrating the two datasets, the records are grouped by ZIP code and month.

This is necessary because multiple Housing or SIRTA records may refer to the same geographic area and time period.

For Housing, the aggregated table counts building job application filing records.

For SIRTA, the aggregated table counts transportation safety records. The exact meaning of these counts depends on the dataset's record structure.

Aggregating the datasets separately prevents a many-to-many merge from artificially increasing the counts.


In [187]:
# Remove records without usable integration keys
# Keep valid date/ZIP records, excluding unverified ZIP 10000

housing_valid = housing_df.dropna(
    subset=["zip_code", "date"]
).copy()

transport_valid = transport_df.dropna(
    subset=["zip_code", "date"]
).copy()

housing_excluded_10000 = (
    housing_valid["zip_code"] == "10000"
).sum()

transport_excluded_10000 = (
    transport_valid["zip_code"] == "10000"
).sum()

housing_valid = housing_valid[
    housing_valid["zip_code"] != "10000"
].copy()

transport_valid = transport_valid[
    transport_valid["zip_code"] != "10000"
].copy()

print("Housing records excluded (ZIP 10000):",
      housing_excluded_10000)

print("SIRTA records excluded (ZIP 10000):",
      transport_excluded_10000)

if housing_valid.empty or transport_valid.empty:
    raise ValueError(
        "At least one dataset has no usable ZIP-date records."
    )

# Identify the overlapping monthly period
start_month = max(
    housing_valid["month"].min(),
    transport_valid["month"].min()
)

end_month = min(
    housing_valid["month"].max(),
    transport_valid["month"].max()
)

if start_month > end_month:
    raise ValueError(
        "No overlapping monthly period between the datasets."
    )

print("Integration period:", start_month, "to", end_month)

# Filter both datasets to the common period
housing_valid = housing_valid[
    housing_valid["month"].between(start_month, end_month)
].copy()

transport_valid = transport_valid[
    transport_valid["month"].between(start_month, end_month)
].copy()

print("Housing records:", len(housing_valid))
print("SIRTA records:", len(transport_valid))

Housing records excluded (ZIP 10000): 163
SIRTA records excluded (ZIP 10000): 44
Integration period: 2022-10 to 2026-06
Housing records: 282748
SIRTA records: 8028


### Geographic ZIP Code Validation

Before integration, ZIP codes were compared against the official NYC ZIP Code Tabulation Areas (ZCTA) reference dataset, which contains 221 geographic areas.

ZIP code `10000` appeared in 163 Housing records and 44 SIRTA records but was not represented in the official ZCTA boundaries.

Because this ZIP could not be verified as a valid geographic area for the proposed ZIP-based analysis, it was excluded from the integration while the original source records were preserved.

Other ZIP codes missing from the ZCTA reference were flagged but not automatically removed because valid USPS ZIP codes do not always have equivalent ZCTA polygons.

This decision helps prevent potentially incorrect geographic matches while limiting unnecessary data loss.

In [188]:
# Aggregate Housing records by ZIP and month
housing_agg = (
    housing_valid
    .groupby(["zip_code", "month"], as_index=False)
    .agg(
        filing_count=("job_base", "nunique"),
        housing_borough=("borough", "first")
    )
)

# Aggregate SIRTA records by ZIP and month
transport_agg = (
    transport_valid
    .groupby(["zip_code", "month"], as_index=False)
    .agg(
        sirta_count=("date", "size"),
        transport_borough=("borough", "first")
    )
)

print("HOUSING AGGREGATION")
print("Shape:", housing_agg.shape)
display(housing_agg.head())

print("\nSIRTA AGGREGATION")
print("Shape:", transport_agg.shape)
display(transport_agg.head())

HOUSING AGGREGATION
Shape: (9059, 4)


,zip_code,month,filing_count,housing_borough
0,10001,2022-10,139,Manhattan
1,10001,2022-11,155,Manhattan
2,10001,2022-12,120,Manhattan
3,10001,2023-01,97,Manhattan
4,10001,2023-02,121,Manhattan



SIRTA AGGREGATION
Shape: (4416, 4)


,zip_code,month,sirta_count,transport_borough
0,10001,2022-10,2,Manhattan
1,10001,2022-11,2,Manhattan
2,10001,2022-12,3,Manhattan
3,10001,2023-01,1,Manhattan
4,10001,2023-02,2,Manhattan


## 3. Dataset Integration

### Integration Process

After aggregation, the Housing and SIRTA datasets are merged using ZIP code and month.

An outer join is used so that geographic and time combinations from both datasets remain visible.

The merge indicator identifies whether each combination appears in both datasets, only in Housing, or only in SIRTA.

A one-to-one validation check ensures that the aggregated tables contain unique ZIP code-month combinations and that the merge does not multiply the records.


In [189]:
# Merge using ZIP code and month
integrated = pd.merge(
    housing_agg,
    transport_agg,
    on=["zip_code", "month"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

print("INTEGRATION COMPLETED")
print("Shape:", integrated.shape)

print("\nMerge coverage:")
print(integrated["_merge"].value_counts())

display(integrated.head(10))

INTEGRATION COMPLETED
Shape: (9068, 7)

Merge coverage:
_merge
left_only     4652
both          4407
right_only       9
Name: count, dtype: int64


,zip_code,month,filing_count,housing_borough,sirta_count,transport_borough,_merge
0,10001,2022-10,139.0,Manhattan,2.0,Manhattan,both
1,10001,2022-11,155.0,Manhattan,2.0,Manhattan,both
2,10001,2022-12,120.0,Manhattan,3.0,Manhattan,both
3,10001,2023-01,97.0,Manhattan,1.0,Manhattan,both
4,10001,2023-02,121.0,Manhattan,2.0,Manhattan,both
5,10001,2023-03,185.0,Manhattan,NaN,<NA>,left_only
6,10001,2023-04,172.0,Manhattan,1.0,Manhattan,both
7,10001,2023-05,141.0,Manhattan,3.0,Manhattan,both
8,10001,2023-06,114.0,Manhattan,2.0,Manhattan,both
9,10001,2023-07,93.0,Manhattan,3.0,Manhattan,both


## 4. Post-Integration Cleaning

### Post-Integration Cleaning

After merging, the integrated dataset is checked for missing values, duplicate keys, inconsistent borough labels, and incorrect data types.

The two borough columns are combined into one standardized attribute.

Missing counts caused by unmatched records are preserved as missing values rather than automatically changed to zero.

A source coverage indicator is retained so that the website and research analyses can distinguish between matched and unmatched observations.

The cleaned dataset is then prepared for validation and visualization.


In [190]:
# Examine missing values
print("Missing values before cleaning:")
print(integrated.isna().sum())

# Identify borough disagreements
borough_conflicts = integrated[
    integrated["housing_borough"].notna()
    & integrated["transport_borough"].notna()
    & (
        integrated["housing_borough"]
        != integrated["transport_borough"]
    )
]

print("\nBorough conflicts:", len(borough_conflicts))
display(borough_conflicts.head())

# Combine borough columns
integrated["borough"] = (
    integrated["housing_borough"]
    .combine_first(integrated["transport_borough"])
)

# Flag conflicting assignments for further review
integrated["borough_conflict"] = (
    integrated.index.isin(borough_conflicts.index)
)

# Remove redundant columns
integrated = integrated.drop(
    columns=["housing_borough", "transport_borough"]
)

# Preserve merge information
integrated = integrated.rename(
    columns={"_merge": "data_coverage"}
)

# Standardize data types
integrated["zip_code"] = integrated["zip_code"].astype("string")

integrated["month"] = integrated["month"].astype("string")

integrated["filing_count"] = (
    integrated["filing_count"].astype("Int64")
)

integrated["sirta_count"] = (
    integrated["sirta_count"].astype("Int64")
)

# Inspect remaining issues
print("\nMissing values after cleaning:")
print(integrated.isna().sum())

print("\nDuplicate ZIP-month keys:")
print(
    integrated.duplicated(
        subset=["zip_code", "month"]
    ).sum()
)

display(integrated.head())

Missing values before cleaning:
zip_code                0
month                   0
filing_count            9
housing_borough         9
sirta_count          4652
transport_borough    4652
_merge                  0
dtype: int64

Borough conflicts: 7


,zip_code,month,filing_count,housing_borough,sirta_count,transport_borough,_merge
3961,10463,2022-11,35.0,Manhattan,1.0,Bronx,both
3965,10463,2023-03,45.0,Manhattan,1.0,Bronx,both
3982,10463,2024-08,40.0,Manhattan,3.0,Bronx,both
6543,11237,2023-02,19.0,Brooklyn,1.0,Queens,both
6582,11237,2026-05,28.0,Brooklyn,1.0,Queens,both



Missing values after cleaning:
zip_code               0
month                  0
filing_count           9
sirta_count         4652
data_coverage          0
borough                0
borough_conflict       0
dtype: int64

Duplicate ZIP-month keys:
0


,zip_code,month,filing_count,sirta_count,data_coverage,borough,borough_conflict
0,10001,2022-10,139,2,both,Manhattan,False
1,10001,2022-11,155,2,both,Manhattan,False
2,10001,2022-12,120,3,both,Manhattan,False
3,10001,2023-01,97,1,both,Manhattan,False
4,10001,2023-02,121,2,both,Manhattan,False


In [191]:
# Inspect all conflicting borough assignments

print("BOROUGH CONFLICTS")

display(
    borough_conflicts[[
        "zip_code",
        "month",
        "housing_borough",
        "transport_borough"
    ]]
)

BOROUGH CONFLICTS


,zip_code,month,housing_borough,transport_borough
3961,10463,2022-11,Manhattan,Bronx
3965,10463,2023-03,Manhattan,Bronx
3982,10463,2024-08,Manhattan,Bronx
6543,11237,2023-02,Brooklyn,Queens
6582,11237,2026-05,Brooklyn,Queens
7791,11385,2022-12,Queens,Brooklyn
7796,11385,2023-05,Queens,Brooklyn


In [192]:
# Investigate suspicious ZIP code 10000

suspect_zip = "10000"

housing_suspect = housing_df[
    housing_df["zip_code"] == suspect_zip
]

transport_suspect = transport_df[
    transport_df["zip_code"] == suspect_zip
]

print("HOUSING RECORDS WITH ZIP 10000")
print("Count:", len(housing_suspect))

display(
    housing_suspect[
        ["postcode", "borough", "filing_date"]
    ].head(10)
)

print("\nSIRTA RECORDS WITH ZIP 10000")
print("Count:", len(transport_suspect))

display(
    transport_suspect[
        ["zip", "borough", "crash_date"]
    ].head(10)
)

HOUSING RECORDS WITH ZIP 10000
Count: 163


,postcode,borough,filing_date
6115,10000.0,Manhattan,2018-03-22 08:32:21
14255,10000.0,Manhattan,2018-08-06 14:02:48
29318,10000.0,Manhattan,2019-04-18 10:18:23
42386,10000.0,Manhattan,2019-09-09 00:00:00
59728,10000.0,Manhattan,2019-12-23 00:00:00
96181,10000.0,Manhattan,2020-09-16
96360,10000.0,Manhattan,2020-09-16
108427,10000.0,Manhattan,2020-12-10
109126,10000.0,Manhattan,2020-12-15
109871,10000.0,Manhattan,2020-12-22 00:00:00



SIRTA RECORDS WITH ZIP 10000
Count: 44


,zip,borough,crash_date
3119,10000.0,Manhattan,2022-10-26
3792,10000.0,Manhattan,2023-05-16
4100,10000.0,Manhattan,2023-06-18
4605,10000.0,Manhattan,2023-08-10
5251,10000.0,Manhattan,2024-01-13
7887,10000.0,Manhattan,2025-03-30
7939,10000.0,Manhattan,2025-04-06
7997,10000.0,Manhattan,2025-04-14
8039,10000.0,Manhattan,2025-04-22
8057,10000.0,Manhattan,2025-04-25


## 5. Integration Validation

### Integration Validation

The integrated dataset is validated through several checks:

1. Verify that ZIP code-month combinations are unique.
2. Confirm that Housing filing counts remain unchanged after merging.
3. Confirm that SIRTA record counts remain unchanged after merging.
4. Check for negative counts and invalid values.
5. Examine matched and unmatched records using an integration coverage chart.
6. Compare monthly activity patterns to identify potential anomalies.

These checks help determine whether the integration preserved the information in both datasets correctly.


In [193]:
# Check for duplicate integration keys
assert not integrated.duplicated(
    subset=["zip_code", "month"]
).any(), "Duplicate integration keys detected!"

# Compare original and integrated Housing counts
assert (
    housing_agg["filing_count"].sum()
    == integrated["filing_count"].sum()
), "Housing counts changed during integration!"

# Compare original and integrated SIRTA counts
assert (
    transport_agg["sirta_count"].sum()
    == integrated["sirta_count"].sum()
), "SIRTA counts changed during integration!"

# Check for negative counts
assert (
    integrated["filing_count"].dropna() >= 0
).all(), "Negative Housing counts detected!"

assert (
    integrated["sirta_count"].dropna() >= 0
).all(), "Negative SIRTA counts detected!"

print("All integration checks passed successfully!")

print("Housing filings:", integrated["filing_count"].sum())
print("SIRTA records:", integrated["sirta_count"].sum())

All integration checks passed successfully!
Housing filings: 282748
SIRTA records: 8028


In [194]:
# Visualize integration coverage
coverage = (
    integrated["data_coverage"]
    .value_counts()
    .reset_index()
)

coverage.columns = ["Coverage", "Count"]

fig = px.bar(
    coverage,
    x="Coverage",
    y="Count",
    color="Coverage",
    title="Housing and SIRTA Integration Coverage",
    labels={
        "Coverage": "Source Coverage",
        "Count": "ZIP-Month Combinations"
    }
)

fig.show()

In [195]:
# Calculate integration coverage percentages

coverage_summary = (
    integrated["data_coverage"]
    .value_counts()
    .reset_index()
)

coverage_summary.columns = [
    "Merge Category",
    "ZIP-Month Combinations"
]

coverage_summary["Percentage"] = (
    coverage_summary["ZIP-Month Combinations"]
    / len(integrated) * 100
).round(2)

display(coverage_summary)

,Merge Category,ZIP-Month Combinations,Percentage
0,left_only,4652,51.3
1,both,4407,48.6
2,right_only,9,0.1


### Integration Coverage Results

After standardizing dates and excluding the unverified ZIP code `10000`, the integration produced **9,068 unique ZIP-month combinations** covering October 2022 to June 2026.

Of these:
- **4,407 (48.60%)** appeared in both datasets.
- **4,652 (51.30%)** appeared only in Housing.
- **9 (0.10%)** appeared only in SIRTA.

The integration preserved the aggregated record counts without creating duplicate ZIP-month combinations.

Seven borough inconsistencies were identified and flagged for investigation. Missing values introduced by unmatched combinations were preserved instead of being automatically replaced with zero.

ZIP code `10000` was excluded provisionally because it could not be verified against the NYC ZCTA reference. Other ZIP codes absent from that reference were flagged rather than automatically removed, as valid postal ZIP codes do not always have matching ZCTA areas.

These results are preliminary because the SIRTA dataset is still undergoing geographic cleaning.

In [196]:
# Calculate missing ZIP statistics

zip_quality = pd.DataFrame({
    "Dataset": ["Housing", "SIRTA"],
    "Total Records": [
        len(housing_df),
        len(transport_df)
    ],
    "Missing ZIP": [
        housing_df["zip_code"].isna().sum(),
        transport_df["zip_code"].isna().sum()
    ]
})

zip_quality["Missing ZIP %"] = (
    zip_quality["Missing ZIP"]
    / zip_quality["Total Records"] * 100
).round(2)

display(zip_quality)

,Dataset,Total Records,Missing ZIP,Missing ZIP %
0,Housing,571131,1469,0.26
1,SIRTA,13309,5237,39.35


In [197]:
# Geographic validation summary

geo_summary = pd.DataFrame({
    "Check": [
        "Housing records with ZIP 10000",
        "SIRTA records with ZIP 10000",
        "Housing ZIPs outside ZCTA reference",
        "SIRTA ZIPs outside ZCTA reference",
        "Borough conflicts after integration"
    ],
    "Count": [
        housing_df["zip_10000_flag"].sum(),
        transport_df["zip_10000_flag"].sum(),
        housing_df["zip_not_in_zcta"].sum(),
        transport_df["zip_not_in_zcta"].sum(),
        integrated["borough_conflict"].sum()
    ]
})

display(geo_summary)

,Check,Count
0,Housing records with ZIP 10000,163
1,SIRTA records with ZIP 10000,44
2,Housing ZIPs outside ZCTA reference,3801
3,SIRTA ZIPs outside ZCTA reference,48
4,Borough conflicts after integration,7


## 6. Research Question 5


**Do housing filings and SIRTA transportation safety records show similar monthly patterns across NYC boroughs?**

This question examines whether changes in housing construction activity are associated with changes in transportation safety records over time.

The analysis uses an interactive line chart to compare monthly activity across boroughs.

A lag-correlation analysis is also conducted to explore whether changes in Housing activity tend to occur before, after, or during similar changes in SIRTA records.

The results are exploratory and do not establish causation.


In [198]:
# Use matched records with consistent geographic data
matched = integrated[
    (integrated["data_coverage"] == "both")
    & (~integrated["borough_conflict"])
    & (integrated["borough"].notna())
].copy()

# Aggregate to borough-month level
monthly_borough = (
    matched
    .groupby(["borough", "month"], as_index=False)
    .agg(
        filing_count=("filing_count", "sum"),
        sirta_count=("sirta_count", "sum")
    )
)

monthly_borough["month"] = pd.to_datetime(
    monthly_borough["month"]
)

print("Borough-month observations:", len(monthly_borough))

display(monthly_borough.head())

Borough-month observations: 225


,borough,month,filing_count,sirta_count
0,Bronx,2022-10-01,369,30
1,Bronx,2022-11-01,474,31
2,Bronx,2022-12-01,294,36
3,Bronx,2023-01-01,396,16
4,Bronx,2023-02-01,305,21


In [199]:
# Reshape for Plotly
chart_data = monthly_borough.melt(
    id_vars=["borough", "month"],
    value_vars=["filing_count", "sirta_count"],
    var_name="dataset",
    value_name="count"
)

# Create interactive monthly chart
fig = px.line(
    chart_data,
    x="month",
    y="count",
    color="dataset",
    facet_row="borough",
    title="Monthly Housing Filings vs SIRTA Records",
    markers=True,
    height=max(
        500,
        monthly_borough["borough"].nunique() * 250
    )
)

fig.update_yaxes(matches=None)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    legend_title="Dataset"
)

fig.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig.show()

In [200]:
lag_results = []

for borough, group in monthly_borough.groupby("borough"):

    group = group.sort_values("month").set_index("month")

    # Preserve missing months as missing values
    group = group.asfreq("MS")

    for lag in range(-3, 4):

        # Positive lag: filings precede SIRTA records
        shifted = group["filing_count"].shift(lag)

        paired = pd.DataFrame({
            "filings": shifted,
            "sirta": group["sirta_count"]
        }).dropna()

        if (
            len(paired) >= 6
            and paired["filings"].nunique() > 1
            and paired["sirta"].nunique() > 1
        ):
            corr = paired["filings"].corr(paired["sirta"])
        else:
            corr = np.nan

        lag_results.append({
            "borough": borough,
            "lag": lag,
            "correlation": corr,
            "observations": len(paired)
        })

lag_df = pd.DataFrame(lag_results)

fig = px.line(
    lag_df,
    x="lag",
    y="correlation",
    color="borough",
    markers=True,
    hover_data=["observations"],
    title="Housing–SIRTA Lag Correlation by Borough",
    labels={
        "lag": "Lag (Months)",
        "correlation": "Pearson Correlation"
    }
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

display(lag_df)

,borough,lag,correlation,observations
0,Bronx,-3,0.064201,42
1,Bronx,-2,0.024306,43
2,Bronx,-1,0.218288,44
3,Bronx,0,0.619275,45
4,Bronx,1,0.278581,44
5,Bronx,2,0.235744,43
6,Bronx,3,0.225693,42
7,Brooklyn,-3,0.221122,42
8,Brooklyn,-2,0.175412,43
9,Brooklyn,-1,0.367999,44


## 7. Research Question 6


**Which ZIP codes have more or fewer SIRTA transportation safety records than expected based on their housing filing activity?**

This question investigates differences in the relationship between housing development activity and transportation safety records across NYC ZIP codes.

A scatter plot is used to compare the two measures.

A simple linear regression model estimates the general relationship between the variables.

Residuals are then calculated to identify ZIP codes with observed SIRTA counts substantially above or below the fitted values.

Large positive residuals indicate higher observed counts than the model estimates, while large negative residuals indicate lower observed counts.

These findings are exploratory and should not be interpreted as direct measures of neighborhood safety. The model does not control for population, traffic exposure, or other important factors.


### Analytical Sample Limitations

Research Questions 5 and 6 currently use ZIP-month combinations that appear in both datasets.

Although this provides comparable observations for both Housing and SIRTA, it excludes locations or months represented in only one dataset. This may affect the observed relationships and regression estimates.

The analysis also uses counts rather than population-adjusted or traffic-adjusted rates.

Consequently, the results should be interpreted as exploratory associations rather than evidence that housing activity directly causes changes in transportation safety.

The analysis will be updated using the final cleaned SIRTA dataset.

In [201]:
# Aggregate matched records by ZIP code
zip_analysis = (
    matched
    .groupby("zip_code", as_index=False)
    .agg(
        filing_count=("filing_count", "sum"),
        sirta_count=("sirta_count", "sum"),
        months_observed=("month", "nunique"),
        borough=("borough", "first")
    )
)

# Keep ZIP codes with at least 3 matched months
zip_analysis = zip_analysis[
    zip_analysis["months_observed"] >= 3
].copy()

# Calculate average activity per observed month
zip_analysis["filings_per_month"] = (
    zip_analysis["filing_count"]
    / zip_analysis["months_observed"]
)

zip_analysis["sirta_per_month"] = (
    zip_analysis["sirta_count"]
    / zip_analysis["months_observed"]
)

display(zip_analysis.head())

,zip_code,filing_count,sirta_count,months_observed,borough,filings_per_month,sirta_per_month
0,10001,4507,71,37,Manhattan,121.810811,1.918919
1,10002,2048,84,37,Manhattan,55.351351,2.27027
2,10003,3931,62,35,Manhattan,112.314286,1.771429
3,10004,160,9,8,Manhattan,20.0,1.125
5,10006,33,3,3,Manhattan,11.0,1.0


In [202]:
# Prepare regression variables
x = zip_analysis["filings_per_month"].to_numpy(
    dtype=float
)

y = zip_analysis["sirta_per_month"].to_numpy(
    dtype=float
)

if len(x) < 3 or np.unique(x).size < 2:
    raise ValueError(
        "Not enough observations for a meaningful regression."
    )

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)

# Predicted SIRTA records per month
zip_analysis["predicted_sirta"] = (
    slope * zip_analysis["filings_per_month"] + intercept
)

# Calculate residuals
zip_analysis["residual"] = (
    zip_analysis["sirta_per_month"]
    - zip_analysis["predicted_sirta"]
)

# Classify the most extreme observed residuals
zip_analysis["category"] = "Other"

top_high = zip_analysis.nlargest(
    5, "residual"
).index

top_low = zip_analysis.nsmallest(
    5, "residual"
).index

zip_analysis.loc[top_high, "category"] = (
    "Higher than estimated"
)

zip_analysis.loc[top_low, "category"] = (
    "Lower than estimated"
)

print("Regression slope:", round(slope, 4))
print("Regression intercept:", round(intercept, 4))

display(
    zip_analysis.sort_values(
        "residual", ascending=False
    ).head(10)
)

Regression slope: 0.0066
Regression intercept: 1.4257


,zip_code,filing_count,sirta_count,months_observed,borough,filings_per_month,sirta_per_month,predicted_sirta,residual,category
125,11236,1423,136,43,Brooklyn,33.093023,3.162791,1.643783,1.519008,Higher than estimated
93,11203,1419,118,42,Brooklyn,33.785714,2.809524,1.648348,1.161176,Higher than estimated
97,11207,2223,128,45,Brooklyn,49.4,2.844444,1.751249,1.093195,Higher than estimated
63,10456,1385,109,40,Bronx,34.625,2.725,1.653879,1.071121,Higher than estimated
123,11234,1586,119,44,Brooklyn,36.045455,2.704545,1.66324,1.041305,Higher than estimated
102,11212,2088,119,43,Brooklyn,48.55814,2.767442,1.745701,1.021741,Other
119,11230,1231,104,40,Brooklyn,30.775,2.6,1.628507,0.971493,Other
74,10467,1553,100,40,Bronx,38.825,2.5,1.681558,0.818442,Other
116,11226,1445,99,41,Brooklyn,35.243902,2.414634,1.657958,0.756676,Other
144,11368,1182,66,27,Queens,43.777778,2.444444,1.714198,0.730247,Other


In [203]:
# Create scatter plot
fig = px.scatter(
    zip_analysis,
    x="filings_per_month",
    y="sirta_per_month",
    color="category",
    hover_name="zip_code",
    hover_data=[
        "borough",
        "months_observed",
        "predicted_sirta",
        "residual"
    ],
    title="Housing Activity vs SIRTA Activity by ZIP Code",
    labels={
        "filings_per_month": "Average Housing Filings per Month",
        "sirta_per_month": "Average SIRTA Records per Month"
    }
)

# Add regression line
x_line = np.linspace(x.min(), x.max(), 100)

fig.add_trace(
    go.Scatter(
        x=x_line,
        y=slope * x_line + intercept,
        mode="lines",
        name="Regression Line",
        line=dict(dash="dash", color="black")
    )
)

fig.update_layout(height=600)

fig.show()

In [204]:
print("TOP 5 ZIP CODES ABOVE THE REGRESSION LINE")

display(
    zip_analysis.nlargest(
        5, "residual"
    )[[
        "zip_code",
        "borough",
        "filings_per_month",
        "sirta_per_month",
        "predicted_sirta",
        "residual"
    ]]
)

print("\nTOP 5 ZIP CODES BELOW THE REGRESSION LINE")

display(
    zip_analysis.nsmallest(
        5, "residual"
    )[[
        "zip_code",
        "borough",
        "filings_per_month",
        "sirta_per_month",
        "predicted_sirta",
        "residual"
    ]]
)

TOP 5 ZIP CODES ABOVE THE REGRESSION LINE


,zip_code,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
125,11236,Brooklyn,33.093023,3.162791,1.643783,1.519008
93,11203,Brooklyn,33.785714,2.809524,1.648348,1.161176
97,11207,Brooklyn,49.4,2.844444,1.751249,1.093195
63,10456,Bronx,34.625,2.725,1.653879,1.071121
123,11234,Brooklyn,36.045455,2.704545,1.66324,1.041305



TOP 5 ZIP CODES BELOW THE REGRESSION LINE


,zip_code,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
12,10014,Manhattan,95.533333,1.2,2.055276,-0.855276
134,11357,Queens,34.466667,1.066667,1.652836,-0.586169
18,10021,Manhattan,78.421053,1.368421,1.942503,-0.574082
57,10314,Staten Island,61.2,1.266667,1.829013,-0.562346
40,10075,Manhattan,44.611111,1.166667,1.719689,-0.553023


## 8. Export Final Integrated Dataset

### Final Integrated Dataset

After completing the integration and validation stages, the integrated dataset is exported as a CSV file.

The final file contains ZIP codes, boroughs, months, Housing filing counts, SIRTA record counts, and source coverage information.

This dataset will be provided to the website development team for interactive filtering, visualization, and reporting.

The integrated file is also retained as an output of the data engineering pipeline to support reproducibility.


In [205]:
# Select final columns
final_integrated = integrated[[
    "zip_code",
    "borough",
    "month",
    "filing_count",
    "sirta_count",
    "data_coverage",
    "borough_conflict"
]].copy()

# Export dataset
filename = "NYC_Integrated_Dataset.csv"

final_integrated.to_csv(
    filename,
    index=False
)

print("Dataset exported successfully!")
print("Final shape:", final_integrated.shape)

display(final_integrated.head())

# Download from Colab
files.download(filename)

Dataset exported successfully!
Final shape: (9068, 7)


,zip_code,borough,month,filing_count,sirta_count,data_coverage,borough_conflict
0,10001,Manhattan,2022-10,139,2,both,False
1,10001,Manhattan,2022-11,155,2,both,False
2,10001,Manhattan,2022-12,120,3,both,False
3,10001,Manhattan,2023-01,97,1,both,False
4,10001,Manhattan,2023-02,121,2,both,False


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [206]:
print("========== MEMBER 3 FINAL CHECK ==========")

print("\n1. DATASET SIZES")
print("Housing records:", len(housing))
print("Crash records:", len(transport))
print("Integrated records:", len(integrated))

print("\n2. INTEGRATION PERIOD")
print("Start:", start_month)
print("End:", end_month)

print("\n3. MERGE COVERAGE")
print(integrated["data_coverage"].value_counts())

print("\n4. MISSING VALUES")
print(integrated.isnull().sum())

print("\n5. DUPLICATE INTEGRATION KEYS")
print(
    integrated.duplicated(
        subset=["zip_code", "month"]
    ).sum()
)

print("\n6. BOROUGH CONFLICTS")
print(integrated["borough_conflict"].sum())

print("\n7. RESEARCH QUESTION DATA")
print("RQ5 borough-month observations:", len(monthly_borough))
print("RQ6 ZIP codes:", len(zip_analysis))

print("\n8. FINAL DATASET")
print("Shape:", final_integrated.shape)
display(final_integrated.head(10))

print("\n========== CHECK COMPLETE ==========")

========== MEMBER 3 FINAL CHECK ==========

1. DATASET SIZES
Housing records: 571131
Crash records: 13309
Integrated records: 9068

2. INTEGRATION PERIOD
Start: 2022-10
End: 2026-06

3. MERGE COVERAGE
data_coverage
left_only     4652
both          4407
right_only       9
Name: count, dtype: int64

4. MISSING VALUES
zip_code               0
month                  0
filing_count           9
sirta_count         4652
data_coverage          0
borough                0
borough_conflict       0
dtype: int64

5. DUPLICATE INTEGRATION KEYS
0

6. BOROUGH CONFLICTS
7

7. RESEARCH QUESTION DATA
RQ5 borough-month observations: 225
RQ6 ZIP codes: 171

8. FINAL DATASET
Shape: (9068, 7)


,zip_code,borough,month,filing_count,sirta_count,data_coverage,borough_conflict
0,10001,Manhattan,2022-10,139,2,both,False
1,10001,Manhattan,2022-11,155,2,both,False
2,10001,Manhattan,2022-12,120,3,both,False
3,10001,Manhattan,2023-01,97,1,both,False
4,10001,Manhattan,2023-02,121,2,both,False
5,10001,Manhattan,2023-03,185,<NA>,left_only,False
6,10001,Manhattan,2023-04,172,1,both,False
7,10001,Manhattan,2023-05,141,3,both,False
8,10001,Manhattan,2023-06,114,2,both,False
9,10001,Manhattan,2023-07,93,3,both,False



========== CHECK COMPLETE ==========
